# Base vs tuned: one complaint, two replies, every check

Runs the pinned base model `Qwen/Qwen3-1.7B` and the public fine-tuned adapter
[`joe-alfaro/groundcite-qwen3-1.7b`](https://huggingface.co/joe-alfaro/groundcite-qwen3-1.7b) on
the same vehicle complaint and reference snippets, then prints both replies side by side with the
result of every deterministic check of the citation contract (one next diagnostic step, exactly
one cited snippet ID that was provided and supports it, no repair as the answer; with no usable
snippet, say so, cite nothing and ask for one missing detail).

It runs logged out: no token, no secrets and no login. Both models are public on the Hugging Face
Hub.

Before running:

- A free GPU (Kaggle **GPU T4 x2** or Colab **T4**) and internet access. Without a GPU it falls
  back to the CPU, which takes minutes per reply.
- `REPO_URL` in the parameters cell is the code repo's clone URL. The hosted copy of this
  notebook has it filled in; the notebook stops, naming `REPO_URL`, while it is empty.
- Edit `COMPLAINT` and `SNIPPETS` to try your own input. `SNIPPETS = []` sends no snippets, so a
  reply that follows the contract says it has no supporting snippet and asks one question.

The snippets are synthetic, not real service information.

In [ ]:
# The code repo's clone URL; the hosted copy of this notebook fills it in.
REPO_URL = ""
# The code commit the notebook installs (the same pin as the training and eval notebooks).
REPO_COMMIT = "549eefc2eb084543427457be5526ad060cc105a8"
# The base model at its train/config.toml pin, and the public adapter at a Hub commit.
BASE_MODEL = "Qwen/Qwen3-1.7B"
BASE_REVISION = "70d244cc86ccca08cf5af4e1e306ecf908b1ad5e"
ADAPTER_REPO = "joe-alfaro/groundcite-qwen3-1.7b"
ADAPTER_REVISION = "b16b566d0d36d06744c90342fe836a4e6decc4e8"
# The input: a free-text complaint and the reference snippets the reply may cite
# ({"id", "title" (optional), "text"}); [] means no snippets.
COMPLAINT = (
    "My 2016 hatchback takes three or four tries to restart after a short stop on a hot day, "
    "but it starts right up every morning."
)
SNIPPETS = [
    {
        "id": "TSB-4417",
        "title": "Hot restart: fuel pressure leak-down test",
        "text": (
            "Applies to gasoline engines with port injection, model years 2012-2019. Connect a "
            "fuel pressure gauge to the rail test port, run the engine to operating temperature, "
            "then switch it off and watch the gauge for 10 minutes. A drop of more than 10 psi "
            "points to a leaking injector or check valve."
        ),
    },
    {
        "id": "SM 4.2.6",
        "title": "Wiper motor park switch check",
        "text": (
            "With the key on and the wiper switch off, back-probe the park switch terminal at the "
            "wiper motor connector. It should read battery voltage until the blades reach the "
            "park position."
        ),
    },
]

Clone the code repo at the pinned commit and install it around the platform's torch (the
CUDA torch stays as it is).

In [ ]:
import os
import subprocess
import sys
import tempfile
from importlib import invalidate_caches, metadata

if not REPO_URL:
    raise ValueError(
        "REPO_URL is empty: set it in the parameters cell to the code repo's clone URL"
    )

REPO_DIR = "/tmp/groundcite-demo"
# The packages the demo needs, at the pins of the training and eval notebooks.
PACKAGES = ("transformers", "peft", "accelerate", "tokenizers", "safetensors", "huggingface-hub")


def sh(cmd: str) -> None:
    subprocess.run(cmd, shell=True, check=True)


sh(f"git clone --quiet {REPO_URL} {REPO_DIR}")
sh(f"git -C {REPO_DIR} checkout --quiet {REPO_COMMIT}")
# The preinstalled torch stack is a constraint, so pip never swaps it out.
pins = []
for name in ("torch", "torchvision", "torchaudio", "triton"):
    try:
        pins.append(f"{name}=={metadata.version(name)}\n")
    except metadata.PackageNotFoundError:
        pass
with tempfile.NamedTemporaryFile("w", suffix=".txt", delete=False) as constraints:
    constraints.writelines(pins)
pattern = "|".join(PACKAGES)
requirements = f"{REPO_DIR}/train/requirements-kaggle.txt"
sh(f"pip install --quiet -c {constraints.name} $(grep -E '^({pattern})==' {requirements})")
sh(f"pip install --quiet --no-deps -e {REPO_DIR}")
# The running kernel does not read the new install's path file, so add the source directly.
sys.path.insert(0, f"{REPO_DIR}/src")
invalidate_caches()

Load the base model and the adapter at their pins (fp16 on the GPU). The adapter is injected
into the base; the base reply is generated with the adapter disabled.

In [ ]:
# Kaggle's "GPU T4 x2" shows two GPUs; the model runs on one.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")

import torch
from peft import PeftModel
from transformers import AutoModelForCausalLM, AutoTokenizer

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.float16 if DEVICE == "cuda" else torch.float32
if DEVICE == "cpu":
    print("No GPU found: running on the CPU, which is slow (minutes per reply).")

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, revision=BASE_REVISION)
base = AutoModelForCausalLM.from_pretrained(BASE_MODEL, revision=BASE_REVISION, dtype=DTYPE)
model = PeftModel.from_pretrained(base.to(DEVICE), ADAPTER_REPO, revision=ADAPTER_REVISION)
model = model.to(DEVICE).eval()

Render the prompt with the project's zero-shot prompt (thinking off), generate greedily, and run
the deterministic checks on both replies.

In [ ]:
import re
import textwrap

from groundcite.checks import run_checks
from groundcite.config import decoding_params
from groundcite.evalset import parse_row
from groundcite.prompts import render_chat_text, render_messages

row = {"id": "demo", "messages": [{"role": "user", "content": COMPLAINT}]}
if SNIPPETS:
    row["snippets"] = SNIPPETS
scenario = parse_row(row)
system, messages = render_messages(scenario, "zero_shot")
prompt = render_chat_text(tokenizer, system, messages)
# The fixed [decoding.hf] parameters from config/models.toml: greedy, capped length.
GENERATION = {**decoding_params(f"hf:{BASE_MODEL}"), "do_sample": False}


def generate_reply() -> str:
    inputs = tokenizer(prompt, return_tensors="pt", add_special_tokens=False).to(model.device)
    with torch.inference_mode():
        output = model.generate(**inputs, **GENERATION)
    new_tokens = output[0][inputs["input_ids"].shape[-1] :]
    reply = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return re.sub(r"<think>.*?</think>", "", reply, flags=re.DOTALL).strip()


with model.disable_adapter():
    base_reply = generate_reply()
tuned_reply = generate_reply()

REPLIES = {"base": base_reply, "tuned": tuned_reply}
CHECK_RESULTS = {name: run_checks(reply, scenario) for name, reply in REPLIES.items()}

In [ ]:
def verdict(passed: bool | None) -> str:
    return "n/a" if passed is None else ("PASS" if passed else "FAIL")


print("COMPLAINT:", COMPLAINT)
print("SNIPPETS:", ", ".join(s["id"] for s in SNIPPETS) or "(none)")
for name, label in (("base", f"BASE  {BASE_MODEL}"), ("tuned", f"TUNED {ADAPTER_REPO}")):
    print()
    print(label)
    print(textwrap.fill(REPLIES[name], width=100, initial_indent="  ", subsequent_indent="  "))

print()
print(f"{'check':<20} {'severity':<9} {'base':<6} {'tuned':<6}")
print("-" * 44)
for base_result, tuned_result in zip(CHECK_RESULTS["base"], CHECK_RESULTS["tuned"], strict=True):
    print(
        f"{base_result.name:<20} {base_result.severity or '':<9} "
        f"{verdict(base_result.passed):<6} {verdict(tuned_result.passed):<6}"
    )
for name in ("base", "tuned"):
    failed = [r for r in CHECK_RESULTS[name] if r.passed is False]
    if failed:
        print()
        print(f"{name} failures:")
        for result in failed:
            print(f"  {result.name}: {result.detail}")